# Embodied AI | Embodied / Physical

In [1]:
# Embodied AI: Grid Navigation with BFS Pathfinding, Obstacles, and Dynamic Replanning
# Pure Python — no LLM needed. Demonstrates sense-plan-act loop with real obstacle avoidance.
from collections import deque

In [2]:
ROWS, COLS = 10, 15

# Static obstacles (walls)
WALLS = {
    (2,3),(2,4),(2,5),(2,6),(2,7),   # horizontal wall
    (4,1),(5,1),(6,1),                # vertical wall
    (5,8),(5,9),(5,10),(5,11),        # another wall
    (7,5),(7,6),(7,7),                # lower wall
}

def bfs(start, goal, blocked):
    """BFS pathfinding on a grid. Returns path as list of (row, col) or None."""
    if start == goal:
        return [start]
    queue = deque([(start, [start])])
    visited = {start}
    while queue:
        (r, c), path = queue.popleft()
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < ROWS and 0 <= nc < COLS and (nr, nc) not in blocked and (nr, nc) not in visited:
                new_path = path + [(nr, nc)]
                if (nr, nc) == goal:
                    return new_path
                visited.add((nr, nc))
                queue.append(((nr, nc), new_path))
    return None  # no path exists

def print_grid(walls, path, agent_pos, goal, dynamic_obs=set()):
    """Print the grid showing walls (#), path (*), agent (A), goal (G), dynamic obstacles (X)."""
    path_set = set(path) if path else set()
    for r in range(ROWS):
        row_str = ""
        for c in range(COLS):
            if (r, c) == agent_pos:
                row_str += "A "
            elif (r, c) == goal:
                row_str += "G "
            elif (r, c) in dynamic_obs:
                row_str += "X "
            elif (r, c) in walls:
                row_str += "# "
            elif (r, c) in path_set:
                row_str += "* "
            else:
                row_str += ". "
        print(row_str)

In [3]:
# --- SCENARIO: Navigate from top-left to bottom-right ---
start, goal = (0, 0), (9, 14)
print("=== Initial Plan ===")
path = bfs(start, goal, WALLS)
assert path is not None, "No path found!"
print_grid(WALLS, path, start, goal)
print(f"Path length: {len(path)} steps\n")

# --- DYNAMIC OBSTACLE appears mid-execution (agent is halfway through) ---
midpoint = len(path) // 2
agent_pos = path[midpoint]
# New obstacle blocks the next few cells on the planned path
DYNAMIC_OBS = {path[midpoint + 1], path[midpoint + 2], path[midpoint + 3]}
all_blocked = WALLS | DYNAMIC_OBS

print(f"=== Dynamic Obstacle at step {midpoint}! Agent at {agent_pos} ===")
print(f"Blocked cells appeared: {DYNAMIC_OBS}")

# Replan from current position
new_path = bfs(agent_pos, goal, all_blocked)
if new_path:
    # Full path = already-traveled + new plan
    full_path = path[:midpoint] + new_path
    print(f"Replanned! New path length: {len(new_path)} steps from current pos")
    print_grid(WALLS, new_path, agent_pos, goal, DYNAMIC_OBS)
    print(f"\nAgent reached goal: {full_path[-1] == goal}")
else:
    print("STUCK: No viable path to goal — requesting human assistance!")
    print_grid(WALLS, [], agent_pos, goal, DYNAMIC_OBS)

=== Initial Plan ===
A . . . . . . . . . . . . . . 
* . . . . . . . . . . . . . . 
* . . # # # # # . . . . . . . 
* . . . . . . . . . . . . . . 
* # . . . . . . . . . . . . . 
* # . . . . . . # # # # . . . 
* # . . . . . . . . . . . . . 
* . . . . # # # . . . . . . . 
* . . . . . . . . . . . . . . 
* * * * * * * * * * * * * * G 
Path length: 24 steps

=== Dynamic Obstacle at step 12! Agent at (9, 3) ===
Blocked cells appeared: {(9, 5), (9, 6), (9, 4)}
Replanned! New path length: 14 steps from current pos
. . . . . . . . . . . . . . . 
. . . . . . . . . . . . . . . 
. . . # # # # # . . . . . . . 
. . . . . . . . . . . . . . . 
. # . . . . . . . . . . . . . 
. # . . . . . . # # # # . . . 
. # . . . . . . . . . . . . . 
. . . . . # # # . . . . . . . 
. . . * * * * * . . . . . . . 
. . . A X X X * * * * * * * G 

Agent reached goal: True
